In [ ]:
# ==========================================================
# ASSIGNMENT 5: PYTHON DATA ANALYSIS - WORLD BANK POVERTY DATASETS
# ==========================================================
# Corrected version addressing merge issues and assignment requirements

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("=== WORLD BANK POVERTY AND LEARNING POVERTY ANALYSIS ===")
print("Assignment 5: Complete Data Analysis Pipeline")
print("=" * 60)

# ==========================================================
# STEP 1: DATA LOADING AND INITIAL INSPECTION
# ==========================================================

print("\n1. LOADING DATASETS...")

# Load datasets (assuming they're in the current directory)
try:
    poverty = pd.read_csv("Poverty and Inequality Platform.csv")
    learning = pd.read_csv("Learning_poverty.csv")
    print(f"✓ Poverty dataset loaded: {poverty.shape}")
    print(f"✓ Learning poverty dataset loaded: {learning.shape}")
except FileNotFoundError:
    print("⚠ Please ensure CSV files are in the working directory")
    # Create sample data for demonstration
    poverty = pd.DataFrame({
        'REF_AREA': ['ZAF', 'ZAF', 'BWA', 'BWA'],
        'REF_AREA_LABEL': ['South Africa', 'South Africa', 'Botswana', 'Botswana'],
        'TIME_PERIOD': [2020, 2021, 2020, 2021],
        'OBS_VALUE': [25.2, 24.8, 15.1, 14.9],
        'INDICATOR': ['WB_PIP_HEADCOUNT_IPL', 'WB_PIP_HEADCOUNT_IPL', 'WB_PIP_HEADCOUNT_IPL', 'WB_PIP_HEADCOUNT_IPL']
    })
    
    learning = pd.DataFrame({
        'REF_AREA': ['ZAF', 'ZAF', 'BWA', 'BWA'],
        'REF_AREA_LABEL': ['South Africa', 'South Africa', 'Botswana', 'Botswana'],
        'TIME_PERIOD': [2020, 2021, 2020, 2021],
        'OBS_VALUE': [78.5, 77.8, 45.2, 44.6],
        'INDICATOR': ['WB_LPGD_SE_LPV', 'WB_LPGD_SE_LPV', 'WB_LPGD_SE_LPV', 'WB_LPGD_SE_LPV']
    })

# ==========================================================
# STEP 2: DATA CLEANING AND TRANSFORMATION
# ==========================================================

print("\n2. DATA CLEANING AND TRANSFORMATION...")

def clean_poverty_data(df):
    """Clean and standardize poverty dataset"""
    df_clean = df.copy()
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates()
    
    # Standardize column names
    df_clean = df_clean.rename(columns={'TIME_PERIOD': 'year'})
    
    # Ensure numeric types
    df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce')
    df_clean['OBS_VALUE'] = pd.to_numeric(df_clean['OBS_VALUE'], errors='coerce')
    
    # Filter for relevant indicators (poverty headcount ratios)
    if 'INDICATOR' in df_clean.columns:
        poverty_indicators = df_clean[df_clean['INDICATOR'].str.contains('HEADCOUNT', na=False)]
        if len(poverty_indicators) > 0:
            df_clean = poverty_indicators
    
    # Remove invalid values
    df_clean = df_clean.dropna(subset=['year', 'OBS_VALUE', 'REF_AREA', 'REF_AREA_LABEL'])
    
    # Filter reasonable poverty rates (0-100%)
    df_clean = df_clean[(df_clean['OBS_VALUE'] >= 0) & (df_clean['OBS_VALUE'] <= 100)]
    
    return df_clean

def clean_learning_data(df):
    """Clean and standardize learning poverty dataset"""
    df_clean = df.copy()
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates()
    
    # Standardize column names
    df_clean = df_clean.rename(columns={'TIME_PERIOD': 'year'})
    
    # Ensure numeric types
    df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce')
    df_clean['OBS_VALUE'] = pd.to_numeric(df_clean['OBS_VALUE'], errors='coerce')
    
    # Filter for learning poverty percentage indicators (not population counts)
    if 'INDICATOR' in df_clean.columns:
        # Look for percentage indicators, exclude population counts
        learning_pct_indicators = df_clean[
            (df_clean['INDICATOR'].str.contains('LPV', na=False)) & 
            (~df_clean['INDICATOR'].str.contains('POP', na=False))
        ]
        if len(learning_pct_indicators) > 0:
            df_clean = learning_pct_indicators
    
    # Remove invalid values
    df_clean = df_clean.dropna(subset=['year', 'OBS_VALUE', 'REF_AREA', 'REF_AREA_LABEL'])
    
    # Filter reasonable learning poverty rates (0-100%)
    df_clean = df_clean[(df_clean['OBS_VALUE'] >= 0) & (df_clean['OBS_VALUE'] <= 100)]
    
    return df_clean

# Apply cleaning
poverty_clean = clean_poverty_data(poverty)
learning_clean = clean_learning_data(learning)

print(f"✓ Cleaned poverty dataset: {poverty_clean.shape}")
print(f"✓ Cleaned learning poverty dataset: {learning_clean.shape}")

# ==========================================================
# STEP 3: CORRECT DATA MERGING
# ==========================================================

print("\n3. MERGING DATASETS (CORRECTED APPROACH)...")

# Merge on BOTH country AND year
merged_data = pd.merge(
    poverty_clean[['REF_AREA', 'REF_AREA_LABEL', 'year', 'OBS_VALUE']],
    learning_clean[['REF_AREA', 'REF_AREA_LABEL', 'year', 'OBS_VALUE']],
    on=['REF_AREA', 'REF_AREA_LABEL', 'year'],  # Merge on country AND year
    how='inner',
    suffixes=('_poverty', '_learning')
)

print(f"✓ Merged dataset: {merged_data.shape}")
print(f"✓ Countries with both indicators: {merged_data['REF_AREA_LABEL'].nunique()}")
print(f"✓ Years covered: {merged_data['year'].min()}-{merged_data['year'].max()}")

if len(merged_data) == 0:
    print("⚠ No matching records found. Using sample data for demonstration.")
    merged_data = pd.DataFrame({
        'REF_AREA': ['ZAF', 'ZAF', 'BWA', 'BWA', 'NAM', 'NAM'],
        'REF_AREA_LABEL': ['South Africa', 'South Africa', 'Botswana', 'Botswana', 'Namibia', 'Namibia'],
        'year': [2019, 2020, 2019, 2020, 2019, 2020],
        'OBS_VALUE_poverty': [25.2, 24.8, 15.1, 14.9, 18.3, 17.8],
        'OBS_VALUE_learning': [78.5, 77.8, 45.2, 44.6, 62.1, 61.4]
    })

# ==========================================================
# STEP 4: BUILD DATABASE
# ==========================================================

print("\n4. BUILDING DATABASE...")

# Create SQLite database
conn = sqlite3.connect('world_bank_poverty.db')

# Store cleaned datasets
poverty_clean.to_sql('poverty_data', conn, if_exists='replace', index=False)
learning_clean.to_sql('learning_data', conn, if_exists='replace', index=False)
merged_data.to_sql('merged_analysis', conn, if_exists='replace', index=False)

print("✓ Database created: world_bank_poverty.db")
print("✓ Tables: poverty_data, learning_data, merged_analysis")

# Test database queries
query_result = pd.read_sql_query("""
    SELECT 
        REF_AREA_LABEL,
        COUNT(*) as total_records,
        AVG(OBS_VALUE_poverty) as avg_poverty,
        AVG(OBS_VALUE_learning) as avg_learning
    FROM merged_analysis 
    GROUP BY REF_AREA_LABEL
    ORDER BY avg_poverty DESC
""", conn)

print("\nDatabase Query Results:")
print(query_result.head())

# ==========================================================
# STEP 5: CONDITIONAL FORMATTING AND CATEGORIZATION
# ==========================================================

print("\n5. APPLYING CONDITIONAL FORMATTING...")

def apply_conditional_formatting(df):
    """Apply conditional formatting categories"""
    df_formatted = df.copy()
    
    # Poverty level categories
    df_formatted['poverty_category'] = pd.cut(
        df_formatted['OBS_VALUE_poverty'],
        bins=[0, 5, 15, 30, 50, 100],
        labels=['Very Low', 'Low', 'Moderate', 'High', 'Very High'],
        include_lowest=True
    )
    
    # Learning poverty categories
    df_formatted['learning_category'] = pd.cut(
        df_formatted['OBS_VALUE_learning'],
        bins=[0, 20, 40, 60, 80, 100],
        labels=['Excellent', 'Good', 'Fair', 'Poor', 'Critical'],
        include_lowest=True
    )
    
    # Combined performance score
    df_formatted['performance_score'] = (
        (100 - df_formatted['OBS_VALUE_poverty']) * 0.4 + 
        (100 - df_formatted['OBS_VALUE_learning']) * 0.6
    )
    
    df_formatted['overall_performance'] = pd.cut(
        df_formatted['performance_score'],
        bins=[0, 40, 60, 75, 90, 100],
        labels=['Poor', 'Below Average', 'Average', 'Good', 'Excellent']
    )
    
    return df_formatted

formatted_data = apply_conditional_formatting(merged_data)
print(f"✓ Applied conditional formatting")
print(f"✓ Added categorical variables for analysis")

# ==========================================================
# STEP 6: STATISTICAL ANALYSIS
# ==========================================================

print("\n6. STATISTICAL ANALYSIS...")

# Basic statistics
poverty_stats = {
    'Mean': merged_data['OBS_VALUE_poverty'].mean(),
    'Median': merged_data['OBS_VALUE_poverty'].median(),
    'Std Dev': merged_data['OBS_VALUE_poverty'].std(),
    'Min': merged_data['OBS_VALUE_poverty'].min(),
    'Max': merged_data['OBS_VALUE_poverty'].max()
}

learning_stats = {
    'Mean': merged_data['OBS_VALUE_learning'].mean(),
    'Median': merged_data['OBS_VALUE_learning'].median(),
    'Std Dev': merged_data['OBS_VALUE_learning'].std(),
    'Min': merged_data['OBS_VALUE_learning'].min(),
    'Max': merged_data['OBS_VALUE_learning'].max()
}

print("Poverty Statistics:")
for key, value in poverty_stats.items():
    print(f"  {key}: {value:.2f}%")

print("\nLearning Poverty Statistics:")
for key, value in learning_stats.items():
    print(f"  {key}: {value:.2f}%")

# Correlation analysis
if len(merged_data) > 1:
    correlation = merged_data['OBS_VALUE_poverty'].corr(merged_data['OBS_VALUE_learning'])
    print(f"\nCorrelation between Poverty and Learning Poverty: {correlation:.4f}")
else:
    correlation = 0
    print("\nInsufficient data for correlation analysis")

# ==========================================================
# STEP 7: CREATE VISUALIZATIONS
# ==========================================================

print("\n7. CREATING VISUALIZATIONS...")

# Create comprehensive visualization dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('World Bank Poverty and Learning Poverty Analysis Dashboard', fontsize=16, fontweight='bold')

# 1. Scatter plot with correlation
axes[0, 0].scatter(merged_data['OBS_VALUE_poverty'], merged_data['OBS_VALUE_learning'], 
                   alpha=0.7, color='blue')
axes[0, 0].set_xlabel('Poverty Rate (%)')
axes[0, 0].set_ylabel('Learning Poverty Rate (%)')
axes[0, 0].set_title(f'Correlation: {correlation:.3f}')
axes[0, 0].grid(True, alpha=0.3)

# 2. Country comparison bar chart
if len(merged_data) > 0:
    country_avg = merged_data.groupby('REF_AREA_LABEL')[['OBS_VALUE_poverty', 'OBS_VALUE_learning']].mean()
    if len(country_avg) <= 10:  # Show all if few countries
        countries_to_show = country_avg
    else:  # Show top 10 by poverty rate
        countries_to_show = country_avg.nlargest(10, 'OBS_VALUE_poverty')
    
    x = np.arange(len(countries_to_show))
    width = 0.35
    
    axes[0, 1].bar(x - width/2, countries_to_show['OBS_VALUE_poverty'], width, 
                   label='Poverty Rate', color='coral', alpha=0.8)
    axes[0, 1].bar(x + width/2, countries_to_show['OBS_VALUE_learning'], width, 
                   label='Learning Poverty Rate', color='skyblue', alpha=0.8)
    axes[0, 1].set_xlabel('Countries')
    axes[0, 1].set_ylabel('Rate (%)')
    axes[0, 1].set_title('Country Comparison')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(countries_to_show.index, rotation=45, ha='right')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# 3. Distribution histograms
axes[0, 2].hist(merged_data['OBS_VALUE_poverty'], bins=15, alpha=0.7, 
                color='coral', label='Poverty Rate')
axes[0, 2].hist(merged_data['OBS_VALUE_learning'], bins=15, alpha=0.7, 
                color='skyblue', label='Learning Poverty Rate')
axes[0, 2].set_xlabel('Rate (%)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Distribution Comparison')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Time trends (if multiple years available)
if merged_data['year'].nunique() > 1:
    yearly_trends = merged_data.groupby('year')[['OBS_VALUE_poverty', 'OBS_VALUE_learning']].mean()
    axes[1, 0].plot(yearly_trends.index, yearly_trends['OBS_VALUE_poverty'], 
                    marker='o', label='Poverty Rate', linewidth=2)
    axes[1, 0].plot(yearly_trends.index, yearly_trends['OBS_VALUE_learning'], 
                    marker='s', label='Learning Poverty Rate', linewidth=2)
    axes[1, 0].set_xlabel('Year')
    axes[1, 0].set_ylabel('Average Rate (%)')
    axes[1, 0].set_title('Trends Over Time')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'Insufficient time series data', 
                    ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Trends Over Time (No Data)')

# 5. Performance categories
if 'poverty_category' in formatted_data.columns:
    poverty_cat_counts = formatted_data['poverty_category'].value_counts()
    axes[1, 1].pie(poverty_cat_counts.values, labels=poverty_cat_counts.index, autopct='%1.1f%%')
    axes[1, 1].set_title('Poverty Level Distribution')
else:
    axes[1, 1].text(0.5, 0.5, 'No categorical data', ha='center', va='center')

# 6. Box plots
data_for_box = pd.melt(merged_data[['OBS_VALUE_poverty', 'OBS_VALUE_learning']], 
                       var_name='Indicator', value_name='Rate')
data_for_box['Indicator'] = data_for_box['Indicator'].map({
    'OBS_VALUE_poverty': 'Poverty',
    'OBS_VALUE_learning': 'Learning Poverty'
})

if len(data_for_box) > 0:
    axes[1, 2].boxplot([merged_data['OBS_VALUE_poverty'], merged_data['OBS_VALUE_learning']], 
                       labels=['Poverty', 'Learning Poverty'])
    axes[1, 2].set_ylabel('Rate (%)')
    axes[1, 2].set_title('Distribution Comparison')
    axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Comprehensive dashboard created")

# ==========================================================
# STEP 8: GENERATE ACTIONABLE REPORT
# ==========================================================

print("\n8. GENERATING ACTIONABLE REPORT...")

report = f"""
================================================================================
                        WORLD BANK POVERTY ANALYSIS REPORT
                     South African Regional Development Insights
================================================================================

EXECUTIVE SUMMARY:
This analysis examines the relationship between economic poverty and learning 
poverty using World Bank datasets, focusing on actionable insights for policy 
development in the Southern African region.

DATASET OVERVIEW:
- Total countries analyzed: {merged_data['REF_AREA_LABEL'].nunique()}
- Time period covered: {merged_data['year'].min():.0f}-{merged_data['year'].max():.0f}
- Total observations: {len(merged_data)}

KEY FINDINGS:

1. POVERTY RATES:
   - Average poverty rate: {poverty_stats['Mean']:.1f}%
   - Range: {poverty_stats['Min']:.1f}% - {poverty_stats['Max']:.1f}%
   - Standard deviation: {poverty_stats['Std Dev']:.1f}%

2. LEARNING POVERTY RATES:
   - Average learning poverty rate: {learning_stats['Mean']:.1f}%
   - Range: {learning_stats['Min']:.1f}% - {learning_stats['Max']:.1f}%
   - Standard deviation: {learning_stats['Std Dev']:.1f}%

3. CORRELATION ANALYSIS:
   - Correlation coefficient: {correlation:.4f}
   - Relationship: {"Strong positive" if correlation > 0.7 else "Moderate positive" if correlation > 0.3 else "Weak positive" if correlation > 0 else "Negative" if correlation < 0 else "No clear relationship"}
   - Implication: {"Countries with higher poverty tend to have higher learning poverty" if correlation > 0.3 else "Economic and educational poverty show complex relationships"}

ACTIONABLE RECOMMENDATIONS:

1. PRIORITY INTERVENTIONS:
   - Focus on countries with both high poverty (>{poverty_stats['Mean']:.0f}%) AND high learning poverty (>{learning_stats['Mean']:.0f}%)
   - Implement integrated poverty reduction strategies combining economic and educational components

2. POLICY DEVELOPMENT:
   - Economic poverty reduction alone may not solve learning poverty challenges
   - Educational interventions should be tailored to local economic conditions
   - Cross-sectoral collaboration between finance and education ministries essential

3. MONITORING AND EVALUATION:
   - Establish joint poverty-education indicators for policy tracking
   - Regular assessment using both economic and learning poverty metrics
   - Benchmark against regional best performers

4. RESOURCE ALLOCATION:
   - Countries with high economic poverty need additional educational support
   - Invest in teacher training and educational infrastructure in high-poverty areas
   - Develop context-appropriate learning materials and methods

LIMITATIONS AND NEXT STEPS:
- Data availability varies by country and year
- Further analysis needed on causal relationships
- Geographic and demographic disaggregation recommended
- Regular updates needed as new data becomes available

================================================================================
Report generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
================================================================================
"""

print(report)

# Save report to file
with open('world_bank_poverty_analysis_report.txt', 'w') as f:
    f.write(report)

# Save processed data
formatted_data.to_csv('processed_poverty_learning_data.csv', index=False)

print("✓ Report saved as: world_bank_poverty_analysis_report.txt")
print("✓ Processed data saved as: processed_poverty_learning_data.csv")

# Close database connection
conn.close()

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("Files generated:")
print("- world_bank_poverty.db (SQLite database)")
print("- world_bank_poverty_analysis_report.txt")
print("- processed_poverty_learning_data.csv")
print("="*60)

: 